# Synthetic spectra and the RAVEL text bridge

This notebook builds a small synthetic binary spectrum with `minato.synthetic`, writes it in the whitespace-delimited text format accepted by RAVEL, and reads it back through `ravel.read_spectra`.

The atmosphere backend below is a tiny analytic example, not a physical model grid. Replace it with an object that implements `get_spectrum(star)` and returns a `Spectrum` or `(wavelength, flux)` tuple.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

os.environ.setdefault("MINATO_QUIET", "1")

from minato.synthetic import BinarySystem, ObservationModel, Spectrum, Star, render_binary
from minato.synthetic.io import write_ravel_txt

## Define a minimal atmosphere backend

MINATO does not hard-code AP18, PoWR, or any local atmosphere-model path. A backend only needs to provide `get_spectrum(star)`.

For a folder of text atmosphere models, start with `TextAtmosphereGrid.from_directory("models/", format="auto")`. It recognises the MINATO convention (`teff25000_logg4.00.txt`) plus common PoWR, TLUSTY, and FASTWIND-style names. For unconventional local names, pass `filename_pattern=...`, pass a parser function, create an editable index with `TextAtmosphereGrid.write_index_template(...)`, or symlink files into the MINATO convention.

In [ ]:
class AnalyticOBGrid:
    def __init__(self):
        self.wavelength = np.linspace(3900.0, 7000.0, 12000)
        self.lines = np.array([4026.0, 4102.0, 4340.0, 4388.0, 4471.0])

    def get_spectrum(self, star):
        flux = np.ones_like(self.wavelength)
        teff_scale = np.clip((35_000.0 - star.teff) / 25_000.0, 0.15, 0.9)
        logg_scale = np.clip((4.6 - star.logg) / 1.2, 0.2, 1.0)
        for index, centre in enumerate(self.lines):
            depth = (0.08 + 0.025 * index) * teff_scale * logg_scale
            width = 0.45 + 0.08 * index
            flux -= depth * np.exp(-0.5 * ((self.wavelength - centre) / width) ** 2)
        return Spectrum(self.wavelength, flux, metadata={"backend": "analytic_ob_grid"})


grid = AnalyticOBGrid()

## Render a binary epoch

For continuum-normalised atmosphere spectra, `render_binary` combines the components with normalised light weights. By default those weights are proportional to `radius**2`, unless `flux_scale` is set on a star.

In [ ]:
system = BinarySystem(
    primary=Star(teff=30_000, logg=4.1, radius=9.0, rv=80.0, vsini=100.0, label="primary"),
    secondary=Star(teff=18_000, logg=4.2, radius=4.5, rv=-140.0, vsini=70.0, label="secondary"),
)

obs = ObservationModel(
    resolving_power=4000,
    snr=50,
    wavelength_min=3900,
    wavelength_max=7000,
    velocity_step=5.0,
    seed=42,
)

spectrum = render_binary(system, atmosphere_grid=grid, observation=obs)
spectrum

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(spectrum.wavelength, spectrum.flux, lw=1.0)
ax.set_xlim(4000, 4600)
ax.set_xlabel("Wavelength (Angstrom)")
ax.set_ylabel("Flux")
plt.show()

## Write and read the RAVEL-compatible text file

In [ ]:
output_dir = Path("synthetic_ravel_example")
output_file = write_ravel_txt(spectrum, output_dir / "synthetic_binary_epoch1.txt")
output_file

In [ ]:
from minato import ravel

wavelengths, fluxes, f_errors, names, jds = ravel.read_spectra(
    [str(output_file)],
    path=str(output_dir),
    file_type="txt",
)

len(wavelengths), names[0], wavelengths[0].shape, fluxes[0].shape, f_errors[0].shape

The file is now in the same text shape used by RAVEL tutorials. A full `SLfit` run is heavier because it runs the probabilistic fitting stack, so leave the following switch off unless you are ready to run the fit.

In [ ]:
RUN_RAVEL_FIT = False

if RUN_RAVEL_FIT:
    lines = [4026, 4102, 4340, 4388, 4471]
    results_path = "synthetic_ravel_results/"
    star_path = ravel.SLfit(
        [str(output_file)],
        data_path=str(output_dir),
        save_path=results_path,
        lines=lines,
        SB2=True,
        file_type="txt",
    )
    print(star_path)